# Mapping the Milky Way Stellar Halo: Discovering the ATLAS-Aliqa Uma Stream
### LSST DA Regional Workshop — Near-Field Cosmology Tutorial

The stellar halo of the Milky Way is a graveyard of disrupted satellite galaxies and globular clusters. As these objects fall into our Galaxy, tidal forces stretch them into long, thin **stellar streams** that trace their orbital paths.

These streams are powerful probes of the Galaxy:
- Their orbits constrain the **shape and mass** of the dark matter halo
- Their stellar populations record the **chemical enrichment history** of early galaxies
- Gaps and wiggles reveal **dark matter substructure**

**The challenge:** stream stars are outnumbered ~1000:1 by foreground Milky Way stars. We beat this by exploiting the fact that stream stars are old, metal-poor, and at a common distance — they trace a distinctive ridge in color-magnitude space called a **stellar isochrone**.

In this tutorial we'll use DELVE DR3 DECam photometry to detect and map the ATLAS-Aliqa Uma stream step by step.

## Setup

In [ ]:
import warnings
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import healpy as hp
import skyproj
import ipywidgets as widgets
from IPython.display import HTML, display, clear_output
from scipy.ndimage import gaussian_filter

import stream_utils as su

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

ISO_DIR = '../data/isochrones/'

## Step 1: Load the data

We're working with **DELVE DR3** — a deep DECam imaging survey covering most of the southern sky. The cutout covers the ATLAS-Aliqa Uma stream region (RA 5°–42°, Dec −42° to −14°) and has already been filtered to point sources with 16 < g < 24.

See the appendix for how to download this or any other region using LSDB.

In [ ]:
cat = pd.read_parquet('../data/atlas_cutout.parquet')

cat['g']     = cat['PSF_MAG_APER_8_G_CORRECTED']
cat['r']     = cat['PSF_MAG_APER_8_R_CORRECTED']
cat['color'] = cat['g'] - cat['r']

print(f'Loaded {len(cat):,} stars')
cat[['RA', 'DEC', 'g', 'r', 'color']].describe()

In [ ]:
atlas_track = np.load('../data/atlas_track.npy')   # shape (N, 2): RA, Dec
aliqa_uma_track = np.load('../data/aliqa_uma_track.npy')

# Add stream coordinates phi1 (along stream) and phi2 (across stream)
cat, R = su.add_stream_coords(cat, atlas_track)

print(f'ATLAS track:     {len(atlas_track)} points')
print(f'Aliqa Uma track: {len(aliqa_uma_track)} points')
print(f'phi1 range: {cat["phi1"].min():.1f} to {cat["phi1"].max():.1f} deg')
print(f'phi2 range: {cat["phi2"].min():.1f} to {cat["phi2"].max():.1f} deg')

## Step 2: Visualize the raw stellar density

Before any filtering, let's see what the raw data looks like — a 2D histogram of all stars on the sky.

**You should not be able to see the stream here.** The ~1000:1 ratio of MW foreground to stream stars completely washes them out. This is precisely the problem the matched filter solves.

In [ ]:
ra_bins  = np.linspace(5, 42, 200)
dec_bins = np.linspace(-42, -14, 150)

counts, _, _ = np.histogram2d(cat['RA'], cat['DEC'], bins=[ra_bins, dec_bins])

fig, ax = plt.subplots(figsize=(12, 6))
img = ax.pcolormesh(
    ra_bins, dec_bins, counts.T,
    cmap='binary', vmin=0, vmax=np.percentile(counts, 98),
    rasterized=True,
)
ax.plot(atlas_track[:, 0], atlas_track[:, 1], 'r-', lw=2, label='ATLAS')
ax.plot(aliqa_uma_track[:, 0], aliqa_uma_track[:, 1], 'b-', lw=2, label='Aliqa Uma')
ax.set_xlim(42, 5)
ax.set_ylim(-42, -14)
ax.set_xlabel('RA (deg)')
ax.set_ylabel('Dec (deg)')
ax.set_title('All stars — unfiltered  (stream tracks shown, but you cannot see them!)')
ax.legend()
plt.colorbar(img, ax=ax, label='stars per bin')
plt.tight_layout()
plt.show()

## Step 3: Build a Hess diagram and fit an isochrone

A **Hess diagram** is a 2D histogram in color-magnitude space. We compute it for an on-stream strip and a background strip (in **stream coordinates** φ₁/φ₂, so the strips align with the stream), then subtract, normalizing by area. The stream's stellar population appears as an overdensity tracing the shape of an old, metal-poor isochrone.

We use **PARSEC isochrones** (Bressan et al. 2012) in the DES/DECam filter system. Three parameters set where the isochrone falls:

| Parameter | Effect on CMD |
|-----------|---------------|
| **Age** | Older → fainter main sequence turnoff |
| **Metallicity Z** | Lower Z → bluer RGB, bluer turnoff |
| **Distance modulus μ** | Larger → shifts everything fainter; μ = 5 log₁₀(d/10 pc) |

The widget below lets you:
1. **Position the on-stream and off-stream strips** — the left panel shows them projected on the sky so you can see they're centred on the stream track. Shift the strips up/down in φ₂, adjust their widths, and toggle the above/below off-stream regions independently to adjust your background selection.
2. **Fit the isochrone** with dropdowns and a slider (right panel).


In [ ]:
clear_output(wait=True)

g_bins     = np.arange(16, 24.1, 0.2)
color_bins = np.arange(-0.2, 1.21, 0.04)

# Build dropdown options from isochrone files present in the repo
# Run scripts/download_isochrones.py to get the full age×Z grid
iso_files   = sorted(f for f in os.listdir(ISO_DIR) if f.endswith('.dat'))
age_options = sorted(set(f.split('_z')[0].replace('iso_a', '') for f in iso_files))
Z_options   = sorted(set(f.split('_z')[1].replace('.dat', '') for f in iso_files))
mu_options  = [f'{mu:.2f}' for mu in np.arange(15.5, 19.1, 0.25)]

PHI1_RANGE  = (-15, 15)
_PHI1_GRID  = np.linspace(*PHI1_RANGE, 300)

_RA_BINS   = np.linspace(5, 42, 200)
_DEC_BINS  = np.linspace(-42, -14, 150)
_SKY_CNTS, _, _ = np.histogram2d(cat['RA'], cat['DEC'],
                                   bins=[_RA_BINS, _DEC_BINS])
_SKY_VMAX  = np.percentile(_SKY_CNTS[_SKY_CNTS > 0], 95)

# Stores the last widget selection so downstream cells pick it up automatically
SELECTED = {
    'phi2_center': 0.0, 'phi2_on_hw': 1.0,
    'phi2_off_sep': 0.5, 'phi2_off_hw': 2.0,
    'use_above': True, 'use_below': True,
    'age': '12.0', 'Z': '0.00030', 'mu': '15.50',
        'C': 0.07, 'E': 2.0, 'gmin': 17.0, 'gmax': 23.5,
}


def _sky_band(phi2_a, phi2_b):
    """Return (ra_poly, dec_poly) for a filled strip between phi2=a and phi2=b."""
    ra_a, dec_a = su.stream_to_icrs(_PHI1_GRID,
                                    np.full_like(_PHI1_GRID, phi2_a), R)
    ra_b, dec_b = su.stream_to_icrs(_PHI1_GRID,
                                    np.full_like(_PHI1_GRID, phi2_b), R)
    return (np.concatenate([ra_a, ra_b[::-1]]),
            np.concatenate([dec_a, dec_b[::-1]]))


def interactive_hess(phi2_center, phi2_on_hw, phi2_off_sep, phi2_off_hw,
                     use_above, use_below, age, Z, mu):
    plt.close('all')
    SELECTED.update(
        phi2_center=phi2_center, phi2_on_hw=phi2_on_hw,
        phi2_off_sep=phi2_off_sep, phi2_off_hw=phi2_off_hw,
        use_above=use_above, use_below=use_below,
        age=age, Z=Z, mu=mu,
    )
    diff, g_b, c_b, on_reg, off_reg = su.compute_hess(
        cat,
        phi1_range=PHI1_RANGE,
        phi2_center=phi2_center,
        phi2_on_hw=phi2_on_hw,
        phi2_off_sep=phi2_off_sep,
        phi2_off_hw=phi2_off_hw,
        use_above=use_above,
        use_below=use_below,
        g_bins=g_bins,
        color_bins=color_bins,
    )
    diff_smooth = gaussian_filter(diff, 0.8)
    vlim = np.nanpercentile(np.abs(diff_smooth), 98)

    fig, axes = plt.subplots(1, 2, figsize=(13, 6),
                             gridspec_kw={'width_ratios': [1.4, 1.6]})

    # --- Left: on-sky selection view ---
    ax_sky = axes[0]
    ax_sky.pcolormesh(_RA_BINS, _DEC_BINS, _SKY_CNTS.T,
                      cmap='Greys', vmin=0, vmax=_SKY_VMAX, shading='auto')

    phi2_on_lo = phi2_center - phi2_on_hw
    phi2_on_hi = phi2_center + phi2_on_hw
    phi2_above_inner = phi2_on_hi + phi2_off_sep
    phi2_above_outer = phi2_above_inner + phi2_off_hw
    phi2_below_inner = phi2_on_lo - phi2_off_sep
    phi2_below_outer = phi2_below_inner - phi2_off_hw

    ra_p, dec_p = _sky_band(phi2_on_lo, phi2_on_hi)
    ax_sky.fill(ra_p, dec_p, color='tomato', alpha=0.45, label='on-stream')

    _off_label_done = False
    for p_lo, p_hi, enabled in [
        (phi2_above_inner, phi2_above_outer, use_above),
        (phi2_below_outer, phi2_below_inner, use_below),
    ]:
        if enabled:
            ra_p, dec_p = _sky_band(p_lo, p_hi)
            ax_sky.fill(ra_p, dec_p, color='deepskyblue', alpha=0.35,
                        label='off-stream' if not _off_label_done else '_')
            _off_label_done = True

    ax_sky.plot(atlas_track[:, 0], atlas_track[:, 1],
                color='gold', lw=1.5, label='ATLAS')
    ax_sky.plot(aliqa_uma_track[:, 0], aliqa_uma_track[:, 1],
                color='orange', lw=1.5, ls='--', label='Aliqa Uma')

    ax_sky.invert_xaxis()
    ax_sky.set_xlabel('RA (deg)', fontsize=12)
    ax_sky.set_ylabel('Dec (deg)', fontsize=12)
    ax_sky.set_title('Selection regions on sky', fontsize=12)
    ax_sky.tick_params(labelsize=11)
    ax_sky.legend(fontsize=10, loc='lower left')

    # --- Right: difference CMD ---
    ax_cmd = axes[1]
    ax_cmd.pcolormesh(color_bins, g_bins, diff_smooth,
                      cmap='RdBu_r', vmin=-vlim, vmax=vlim, shading='auto')
    ax_cmd.invert_yaxis()
    ax_cmd.set_xlabel('g − r', fontsize=12)
    ax_cmd.set_ylabel('g (mag)', fontsize=12)
    ax_cmd.set_xlim(-0.1, 1.0)
    ax_cmd.tick_params(labelsize=11)
    off_str = f'{off_reg["area_deg2"]:.1f} deg²' if off_reg['area_deg2'] > 0 else 'none'
    ax_cmd.set_title(f'On-stream minus off-stream CMD\n'
                     f'on: {on_reg["area_deg2"]:.1f} deg²  off: {off_str}',
                     fontsize=12)

    iso_path = os.path.join(ISO_DIR, f'iso_a{age}_z{Z}.dat')
    if os.path.exists(iso_path):
        su.overplot_isochrone(ax_cmd, ISO_DIR, age, Z, float(mu), g_bins,
                              color='k', lw=2,
                              label=f'age={age} Gyr, Z={Z}, μ={mu}')
        ax_cmd.legend(fontsize=10)
    else:
        ax_cmd.text(0.5, 0.5,
                    f'No file for age={age} Gyr, Z={Z}\n'
                    f'Run scripts/download_isochrones.py',
                    transform=ax_cmd.transAxes, color='red',
                    ha='center', va='center', fontsize=11)

    plt.tight_layout()
    display(fig)
    plt.close(fig)


_w_phi2c  = widgets.FloatSlider(
    min=-3.0, max=3.0, step=0.25, value=0.0,
    description='φ₂ center (deg)',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'),
    continuous_update=False)
_w_onhw   = widgets.FloatSlider(
    min=0.5, max=3.0, step=0.25, value=1.0,
    description='On-stream width ±(deg)',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'),
    continuous_update=False)
_w_offsep = widgets.FloatSlider(
    min=0.0, max=2.0, step=0.25, value=0.5,
    description='Off-stream gap (deg)',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'),
    continuous_update=False)
_w_offhw  = widgets.FloatSlider(
    min=0.5, max=4.0, step=0.5, value=2.0,
    description='Off-stream width (deg)',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'),
    continuous_update=False)
_w_above  = widgets.Checkbox(value=True,  description='Use above-stream background')
_w_below  = widgets.Checkbox(value=True,  description='Use below-stream background')
_w_age    = widgets.Dropdown(options=age_options, value='12.0', description='Age (Gyr)')
_w_Z      = widgets.Dropdown(options=Z_options,   value='0.00030', description='Z')
_w_mu     = widgets.SelectionSlider(
    options=mu_options, value='15.50',
    description='μ (dist. mod.)',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'),
    continuous_update=False)

_out_hess = widgets.Output()

def _update_hess(change=None):
    _out_hess.clear_output(wait=True)
    with _out_hess:
        interactive_hess(
            phi2_center=_w_phi2c.value,  phi2_on_hw=_w_onhw.value,
            phi2_off_sep=_w_offsep.value, phi2_off_hw=_w_offhw.value,
            use_above=_w_above.value,    use_below=_w_below.value,
            age=_w_age.value, Z=_w_Z.value, mu=_w_mu.value,
        )

for _ww in [_w_phi2c, _w_onhw, _w_offsep, _w_offhw,
            _w_above, _w_below, _w_age, _w_Z, _w_mu]:
    _ww.observe(_update_hess, names='value')

_update_hess()   # initial render

display(widgets.VBox([
    _w_phi2c, _w_onhw, _w_offsep, _w_offhw,
    widgets.HBox([_w_above, _w_below]),
    _w_age, _w_Z, _w_mu,
    _out_hess,
]))

## Step 4: Apply the matched filter

A **matched filter** selects stars within a color window around the best-fit isochrone. The window width accounts for photometric uncertainty — it widens at fainter magnitudes where errors are larger:

$$\Delta\mathrm{color}(g) = C + E \times \sigma_g(g)$$

where $\sigma_g(g)$ is the DES photometric error model.

This suppresses ~90% of MW foreground while retaining most stream members.

**Filter parameters** (adjustable in the widget above):

| Parameter | Default | What it controls |
|-----------|---------|-----------------|
| **C** | 0.07 | Fixed symmetric half-width of the color window. Wider → more stars pass but more foreground contamination. |
| **E** | 2.0 | How much the window grows with photometric error. Higher → more permissive at faint magnitudes. |
| **g_min** | 17.0 | Bright magnitude cut — stars brighter than this are excluded from the filter. |
| **g_max** | 23.5 | Faint magnitude cut — stars fainter than this are excluded. |

> **Think about it:** Why is it a good idea to set g_min > 16? What kinds of stars might pass the color filter at the bright end?

**When you're happy with the isochrone fit and filter window above, run the cell below** — it will automatically use the parameters you selected in the widget.

In [ ]:
clear_output(wait=True)

# Precompute the difference Hess diagram once using the strips from Step 3.
_diff_f, _, _, _, _ = su.compute_hess(
    cat,
    phi1_range=PHI1_RANGE,
    phi2_center=SELECTED['phi2_center'],
    phi2_on_hw=SELECTED['phi2_on_hw'],
    phi2_off_sep=SELECTED['phi2_off_sep'],
    phi2_off_hw=SELECTED['phi2_off_hw'],
    use_above=SELECTED['use_above'],
    use_below=SELECTED['use_below'],
    g_bins=g_bins, color_bins=color_bins,
)
_diff_f_smooth = gaussian_filter(_diff_f, 0.8)
_vlim_f = np.nanpercentile(np.abs(_diff_f_smooth), 98)


def show_filter_window(C, E, gmin, gmax):
    plt.close('all')
    SELECTED.update(C=C, E=E, gmin=gmin, gmax=gmax)

    age, Z, mu = SELECTED['age'], SELECTED['Z'], float(SELECTED['mu'])

    fig, ax = plt.subplots(figsize=(6, 8))
    ax.pcolormesh(color_bins, g_bins, _diff_f_smooth,
                  cmap='RdBu_r', vmin=-_vlim_f, vmax=_vlim_f, shading='auto')

    iso_path = os.path.join(ISO_DIR, f'iso_a{age}_z{Z}.dat')
    if os.path.exists(iso_path):
        g_grid, blue_edge, red_edge = su.filter_window_edges(
            ISO_DIR, age, Z, mu, g_bins, C=C, E=E, gmin=gmin, gmax=gmax)
        ax.fill_betweenx(g_grid, blue_edge, red_edge,
                         color='limegreen', alpha=0.3, label='filter window')
        su.overplot_isochrone(ax, ISO_DIR, age, Z, mu, g_bins,
                              color='k', lw=2,
                              label=f'{age} Gyr, Z={Z}, μ={mu}')
        ax.legend(fontsize=10)
    else:
        ax.text(0.5, 0.5, f'No file for age={age}, Z={Z}\nRun scripts/download_isochrones.py',
                transform=ax.transAxes, color='red', ha='center', va='center', fontsize=11)

    ax.invert_yaxis()
    ax.set_xlim(-0.1, 1.0)
    ax.set_xlabel('g − r', fontsize=12)
    ax.set_ylabel('g (mag)', fontsize=12)
    ax.set_title('Matched filter window on difference CMD', fontsize=12)
    ax.tick_params(labelsize=11)
    plt.tight_layout()
    display(fig)
    plt.close(fig)


_sl = {'description_width': 'initial'}
_sw = widgets.Layout(width='450px')

_w_fC    = widgets.FloatSlider(min=0.01, max=0.20, step=0.01, value=0.07,
    description='C (color half-width)', style=_sl, layout=_sw, continuous_update=False)
_w_fE    = widgets.FloatSlider(min=0.0, max=5.0, step=0.25, value=2.0,
    description='E (error scale)', style=_sl, layout=_sw, continuous_update=False)
_w_fgmin = widgets.FloatSlider(min=16.0, max=21.0, step=0.5, value=17.0,
    description='g_min (bright cut)', style=_sl, layout=_sw, continuous_update=False)
_w_fgmax = widgets.FloatSlider(min=20.0, max=24.0, step=0.5, value=23.5,
    description='g_max (faint cut)', style=_sl, layout=_sw, continuous_update=False)

_out_filter = widgets.Output()

def _update_filter(change=None):
    _out_filter.clear_output(wait=True)
    with _out_filter:
        show_filter_window(
            C=_w_fC.value, E=_w_fE.value,
            gmin=_w_fgmin.value, gmax=_w_fgmax.value,
        )

for _ww in [_w_fC, _w_fE, _w_fgmin, _w_fgmax]:
    _ww.observe(_update_filter, names='value')

_update_filter()

display(widgets.VBox([
    _w_fC, _w_fE, _w_fgmin, _w_fgmax,
    _out_filter,
]))

In [ ]:
# Apply matched filter using all parameters selected above — re-run after adjusting either widget
BEST_AGE  = SELECTED['age']
BEST_Z    = SELECTED['Z']
BEST_MU   = float(SELECTED['mu'])
BEST_C    = SELECTED['C']
BEST_E    = SELECTED['E']
BEST_GMIN = SELECTED['gmin']
BEST_GMAX = SELECTED['gmax']

mf_mask  = su.apply_matched_filter(
    cat, ISO_DIR, BEST_AGE, BEST_Z, BEST_MU,
    C=BEST_C, E=BEST_E, gmin=BEST_GMIN, gmax=BEST_GMAX,
)
selected = cat[mf_mask].copy()
print(f'Isochrone:  age={BEST_AGE} Gyr, Z={BEST_Z}, μ={BEST_MU}')
print(f'Filter:     C={BEST_C:.2f}, E={BEST_E:.1f}, g=[{BEST_GMIN:.1f}, {BEST_GMAX:.1f}]')
print(f'Selected:   {mf_mask.sum():,} / {len(cat):,} stars  ({100*mf_mask.mean():.1f}%)')

## Step 5: Map the filtered stars

We bin the matched-filter-selected stars into a **HEALPix** map — a pixelization of the sphere with equal-area pixels. After Gaussian smoothing, the ATLAS-Aliqa Uma stream should stand out clearly.

Use the sliders below to tune the smoothing scale and stretch the colormap until the stream is visible.

In [ ]:
clear_output(wait=True)

NSIDE    = 512
FWHM_DEG = 0.3   # default; also used by the distance-scan animation in Step 6

_MAP_CMAPS = ['bone_r', 'viridis', 'plasma', 'Blues_r', 'YlGn', 'hot_r']


def show_map(fwhm_deg, vmin_pct, vmax_pct, cmap,
             show_atlas, show_aliqa_uma):
    plt.close('all')
    hpxmap = su.stars_to_hpxmap(selected, NSIDE, fwhm_deg)
    vals   = hpxmap[hpxmap > 0]
    vmin   = np.percentile(vals, vmin_pct) if (len(vals) > 0 and vmin_pct > 0) else 0.0
    vmax   = np.percentile(vals, vmax_pct) if len(vals) > 0 else 1.0

    fig, ax = plt.subplots(figsize=(12, 6))
    sp = skyproj.McBrydeSkyproj(
        ax=ax, lon_0=24, extent=[42, 5, -42, -14],
        longitude_ticks='symmetric')
    sp.draw_hpxmap(hpxmap, nest=True, zoom=False,
                   vmin=vmin, vmax=vmax, cmap=cmap)
    if show_atlas:
        sp.ax.plot(atlas_track[:, 0], atlas_track[:, 1],
                   'r-', lw=1.5, label='ATLAS')
    if show_aliqa_uma:
        sp.ax.plot(aliqa_uma_track[:, 0], aliqa_uma_track[:, 1],
                   'b-', lw=1.5, label='Aliqa Uma')
    ax.set_title(
        f'Matched filter map — age={BEST_AGE} Gyr, Z={BEST_Z}, μ={BEST_MU}',
        fontsize=12)
    if show_atlas or show_aliqa_uma:
        ax.legend(fontsize=10)
    plt.tight_layout()
    display(fig)
    plt.close(fig)


_w_fwhm   = widgets.FloatSlider(
    min=0.1, max=0.8, step=0.05, value=0.4,
    description='Smoothing (deg)',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='400px'),
    continuous_update=False)
_w_vmin   = widgets.IntSlider(
    min=0, max=50, step=1, value=10,
    description='vmin percentile',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='400px'),
    continuous_update=False)
_w_vmax   = widgets.IntSlider(
    min=90, max=99, step=1, value=97,
    description='vmax percentile',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='400px'),
    continuous_update=False)
_w_cmap   = widgets.Dropdown(options=_MAP_CMAPS, value='bone_r', description='Colormap')
_w_atlas  = widgets.Checkbox(value=True, description='Show ATLAS track')
_w_aliqa_uma  = widgets.Checkbox(value=True, description='Show Aliqa Uma track')

_out_map = widgets.Output()

def _update_map(change=None):
    _out_map.clear_output(wait=True)
    with _out_map:
        show_map(
            fwhm_deg=_w_fwhm.value, vmin_pct=_w_vmin.value, vmax_pct=_w_vmax.value,
            cmap=_w_cmap.value, show_atlas=_w_atlas.value, show_aliqa_uma=_w_aliqa_uma.value,
        )

for _ww in [_w_fwhm, _w_vmin, _w_vmax, _w_cmap, _w_atlas, _w_aliqa_uma]:
    _ww.observe(_update_map, names='value')

_update_map()   # initial render

display(widgets.VBox([
    _w_fwhm, _w_vmin, _w_vmax, _w_cmap,
    widgets.HBox([_w_atlas, _w_aliqa_uma]),
    _out_map,
]))

## Step 6: Scan across distances

One of the most powerful aspects of the matched filter is that we can sweep across distance moduli, probing different depths in the halo. At the right distance, the stream pops out.

We animate a sweep from μ = 15.5 to 18.0 and watch the ATLAS-Aliqa Uma stream appear and disappear.

In [ ]:
MU_RANGE = np.arange(15.5, 19.1, 0.25)

# Pick up smoothing from the map widget so movie matches the spatial plot
_anim_fwhm = _w_fwhm.value

print(f'Precomputing {len(MU_RANGE)} maps  (smoothing = {_anim_fwhm:.2f}°)...')
mu_maps = {}
for mu in MU_RANGE:
    mask = su.apply_matched_filter(cat, ISO_DIR, BEST_AGE, BEST_Z, mu,
                                   C=BEST_C, E=BEST_E, gmin=BEST_GMIN, gmax=BEST_GMAX)
    mu_maps[mu] = su.stars_to_hpxmap(cat[mask], NSIDE, _anim_fwhm)
print('Done.')

In [ ]:
ref_mu    = MU_RANGE[np.argmin(np.abs(MU_RANGE - BEST_MU))]
ref_vals  = mu_maps[ref_mu][mu_maps[ref_mu] > 0]

# Match colormap and stretch to the map widget above
_anim_vmin = np.percentile(ref_vals, _w_vmin.value) if (len(ref_vals) > 0 and _w_vmin.value > 0) else 0.0
_anim_vmax = np.percentile(ref_vals, _w_vmax.value) if len(ref_vals) > 0 else 1.0
_anim_cmap = _w_cmap.value

_anim_proj = hp.projector.CartesianProj(
    rot=[24, -28, 0], xsize=1000, ysize=500,
    lonra=[-20, 20], latra=[-14, 14],
)
_frame_imgs = {
    mu: _anim_proj.projmap(
        hp.reorder(mu_maps[mu], n2r=True),
        vec2pix_func=lambda x, y, z: hp.vec2pix(NSIDE, x, y, z),
    )
    for mu in MU_RANGE
}

fig, ax = plt.subplots(figsize=(12, 6), facecolor='k')
ax.set_facecolor('k')

def draw_frame(mu):
    ax.cla()
    ax.set_facecolor('k')
    ax.imshow(_frame_imgs[mu], origin='lower', aspect='auto',
              extent=[44, 4, -42, -14],
              vmin=_anim_vmin, vmax=_anim_vmax, cmap=_anim_cmap,
              interpolation='nearest')
    ax.set_xlabel('RA (deg)', fontsize=11)
    ax.set_ylabel('Dec (deg)', fontsize=11)
    ax.tick_params(labelsize=10)
    d_kpc = 10 ** (mu / 5 + 1) / 1000
    ax.set_title(f'μ = {mu:.2f}   (d = {d_kpc:.1f} kpc)', fontsize=13)
    ax.set_xlim(44, 4)    # RA increases to the left (astronomical convention)
    ax.set_ylim(-42, -14)
    return ax,

anim = animation.FuncAnimation(
    fig, draw_frame, frames=MU_RANGE, interval=500, blit=False)
html  = anim.to_jshtml()   # render all frames before closing
plt.close(fig)
HTML(html)

## Step 7: RGB distance map

We can pack three distance ranges into a single **RGB image**. Each color accumulates stars from all matched-filter maps within a μ range:

- **Red** — farther (large μ range)
- **Green** — the ATLAS-Aliqa Uma distance (mid μ range)
- **Blue** — closer (small μ range)

Stars at the stream distance glow green. Structures at other depths appear in red or blue, so you can see whether features shift spatially between shells.

Use the range sliders to set the boundaries of each color shell and the smoothing scale. The maps are pre-computed so only smoothing + projection runs on each update — should be a few seconds.


In [ ]:
NSIDE_RGB = 256

# Pre-compute one unsmoothed star-count map per distance slice.
print(f'Precomputing {len(MU_RANGE)} raw maps for RGB stacking...')
_mu_maps_rgb = {}
for _mu in MU_RANGE:
    _key  = f'{_mu:.2f}'
    _mask = su.apply_matched_filter(cat, ISO_DIR, BEST_AGE, BEST_Z, _mu,
                                    C=BEST_C, E=BEST_E, gmin=BEST_GMIN, gmax=BEST_GMAX)
    _ipix = hp.ang2pix(NSIDE_RGB,
                       cat['RA'].values[_mask], cat['DEC'].values[_mask],
                       lonlat=True, nest=True)
    _hpx  = np.zeros(hp.nside2npix(NSIDE_RGB))
    np.add.at(_hpx, _ipix, 1)
    _mu_maps_rgb[_key] = _hpx
print('Done.')

_rgb_proj = hp.projector.CartesianProj(
    rot=[24, -28, 0], xsize=900, ysize=450,
    lonra=[-20, 20], latra=[-14, 14],
)

In [ ]:
clear_output(wait=True)

mu_str_opts = [f'{mu:.2f}' for mu in MU_RANGE]


def _stack_channel(mu_range, fwhm_deg, vmin_pct, vmax_pct):
    """Sum raw maps within (lo, hi) string range, smooth, and normalize."""
    lo, hi = float(mu_range[0]), float(mu_range[1])
    keys_in = [k for k in mu_str_opts if lo - 1e-9 <= float(k) <= hi + 1e-9]
    stacked = (
        sum(_mu_maps_rgb[k] for k in keys_in)
        if keys_in else np.zeros(hp.nside2npix(NSIDE_RGB))
    )
    ring     = hp.reorder(stacked.astype(float), n2r=True)
    smoothed = hp.smoothing(ring, fwhm=np.radians(fwhm_deg))
    img      = _rgb_proj.projmap(
        smoothed,
        vec2pix_func=lambda x, y, z: hp.vec2pix(NSIDE_RGB, x, y, z),
    )
    return su.normalize_channel(img, lo=vmin_pct, hi=vmax_pct)


def show_rgb(mu_r, mu_g, mu_b, fwhm_deg, vmin_pct, vmax_pct,
             show_atlas, show_aliqa_uma):
    plt.close('all')
    print('Building channels...', end='', flush=True)
    r = _stack_channel(mu_r, fwhm_deg, vmin_pct, vmax_pct); print(' R', end='', flush=True)
    g = _stack_channel(mu_g, fwhm_deg, vmin_pct, vmax_pct); print(' G', end='', flush=True)
    b = _stack_channel(mu_b, fwhm_deg, vmin_pct, vmax_pct); print(' B  done.')

    rgb = np.dstack([r, g, b])

    def _d(mu_str):
        return 10 ** (float(mu_str) / 5 + 1) / 1000

    r_lo, r_hi = mu_r
    g_lo, g_hi = mu_g
    b_lo, b_hi = mu_b

    fig, ax = plt.subplots(figsize=(14, 7), facecolor='k')
    ax.imshow(rgb, origin='lower', aspect='auto', interpolation='bilinear',
              extent=[44, 4, -42, -14])
    if show_atlas:
        ax.plot(atlas_track[:, 0], atlas_track[:, 1],
                'r-', lw=1.5, alpha=0.7, label='ATLAS')
    if show_aliqa_uma:
        ax.plot(aliqa_uma_track[:, 0], aliqa_uma_track[:, 1],
                color='cyan', lw=1.5, ls='--', alpha=0.7, label='Aliqa Uma')
    if show_atlas or show_aliqa_uma:
        ax.legend(fontsize=10, loc='upper right')
    ax.set_facecolor('k')
    ax.set_xlabel('RA (deg)', fontsize=12, color='w')
    ax.set_ylabel('Dec (deg)', fontsize=12, color='w')
    ax.tick_params(labelsize=11, colors='w')
    for spine in ax.spines.values():
        spine.set_edgecolor('w')
    ax.set_title(
        'RGB distance map   (red=far, blue=close)\n'
        f'R: μ={r_lo}–{r_hi} ({_d(r_lo):.0f}–{_d(r_hi):.0f} kpc)   '
        f'G: μ={g_lo}–{g_hi} ({_d(g_lo):.0f}–{_d(g_hi):.0f} kpc)   '
        f'B: μ={b_lo}–{b_hi} ({_d(b_lo):.0f}–{_d(b_hi):.0f} kpc)',
        color='w', fontsize=11,
    )
    ax.set_xlim(44, 4)    # RA increases to the left (matches animation)
    ax.set_ylim(-42, -14)
    plt.tight_layout()
    display(fig)
    plt.close(fig)


_rs = {'description_width': '120px'}
_rl = widgets.Layout(width='580px')
_w_mu_r   = widgets.SelectionRangeSlider(
    options=mu_str_opts, index=(7, 10),
    description='R (far) μ', style=_rs, layout=_rl, continuous_update=False)
_w_mu_g   = widgets.SelectionRangeSlider(
    options=mu_str_opts, index=(4, 6),
    description='G (mid) μ', style=_rs, layout=_rl, continuous_update=False)
_w_mu_b   = widgets.SelectionRangeSlider(
    options=mu_str_opts, index=(0, 3),
    description='B (close) μ', style=_rs, layout=_rl, continuous_update=False)
_w_fwhm_r = widgets.FloatSlider(
    min=0.2, max=1.0, step=0.1, value=0.4,
    description='Smoothing (°)',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='450px'),
    continuous_update=False)
_w_vmin_r = widgets.FloatSlider(
    min=0, max=20, step=0.5, value=1,
    description='vmin percentile',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='450px'),
    continuous_update=False)
_w_vmax_r = widgets.FloatSlider(
    min=80, max=100, step=0.5, value=99.5,
    description='vmax percentile',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='450px'),
    continuous_update=False)
_w_atl_r  = widgets.Checkbox(value=True, description='Show ATLAS track')
_w_alquma_r  = widgets.Checkbox(value=True, description='Show Aliqa Uma track')

_out_rgb = widgets.Output()

def _update_rgb(change=None):
    _out_rgb.clear_output(wait=True)
    with _out_rgb:
        show_rgb(
            mu_r=_w_mu_r.value, mu_g=_w_mu_g.value, mu_b=_w_mu_b.value,
            fwhm_deg=_w_fwhm_r.value, vmin_pct=_w_vmin_r.value, vmax_pct=_w_vmax_r.value,
            show_atlas=_w_atl_r.value, show_aliqa_uma=_w_alquma_r.value,
        )

for _ww in [_w_mu_r, _w_mu_g, _w_mu_b, _w_fwhm_r, _w_vmin_r, _w_vmax_r,
            _w_atl_r, _w_alquma_r]:
    _ww.observe(_update_rgb, names='value')

_update_rgb()   # initial render

display(widgets.VBox([
    _w_mu_r, _w_mu_g, _w_mu_b,
    _w_fwhm_r, _w_vmin_r, _w_vmax_r,
    widgets.HBox([_w_atl_r, _w_alquma_r]),
    _out_rgb,
]))

## Step 8 (bonus): Finding the Elqui Stream

Hidden in this same dataset is **Elqui**, one of DES's most distant known stellar streams — at roughly **50 kpc** (distance modulus μ ≈ 18.5), almost twice as far as ATLAS. It runs through this same region of sky, but because its stars are so faint, the default filter tuned for ATLAS completely misses it. The main-sequence turnoff falls near g ≈ 22 — go back to the widgets and adjust the parameters to find it.

*Does it appear in the RGB map? Which color channel does it light up in?*

## Step 9 (bonus): Download and map your own region

LSDB (Large Survey DataBase) makes it easy to pull spatial cutouts from large HEALPix-partitioned catalogs hosted locally or on the web. The DELVE DR3 catalog is publicly accessible at `data.lsdb.io` — no cluster account needed.

### Download a box (RA × Dec)

```python
import lsdb

CATALOG_URL = 'https://data.lsdb.io/hats/delve/delve_dr3_gold'
COLUMNS = [
    'RA', 'DEC',
    'PSF_MAG_APER_8_G_CORRECTED', 'PSF_MAG_ERR_APER_8_G',
    'PSF_MAG_APER_8_R_CORRECTED', 'PSF_MAG_ERR_APER_8_R',
    'EXT_XGB',
]

df = lsdb.open_catalog(
    CATALOG_URL,
    search_filter=lsdb.BoxSearch(ra=(5, 42), dec=(-42, -14)),
    columns=COLUMNS,
).compute()

df = df[
    (df['EXT_XGB'] == 0) &
    (df['PSF_MAG_APER_8_G_CORRECTED'] > 16) &
    (df['PSF_MAG_APER_8_G_CORRECTED'] < 24)
]
df.to_parquet('my_cutout.parquet')
```

### Download a cone (RA, Dec, radius)

```python
import astropy.units as u

df = lsdb.open_catalog(
    CATALOG_URL,
    search_filter=lsdb.ConeSearch(
        ra=20.0, dec=-27.0,
        radius_arcsec=(10 * u.deg).to_value(u.arcsec),
    ),
    columns=COLUMNS,
).compute()
```

The runnable cells below download a fresh region from the public LSDB endpoint and produce an RGB distance map. Edit the coordinate box in the first cell, then run all three in sequence.

In [ ]:
# ── Step 9 is disabled by default (it downloads data over the network). ──────
# ── Change the line below to RUN_STEP_9 = True to enable it.                 ─
RUN_STEP_9 = False
# ─────────────────────────────────────────────────────────────────────────────
if not RUN_STEP_9:
    raise RuntimeError(
        'Step 9 is disabled by default (requires a network download).\n'
        'Set RUN_STEP_9 = True at the top of this cell to run it.'
    )

import lsdb

CATALOG_URL = 'https://data.lsdb.io/hats/delve/delve_dr3_gold'
COLUMNS = [
    'RA', 'DEC',
    'PSF_MAG_APER_8_G_CORRECTED', 'PSF_MAG_ERR_APER_8_G',
    'PSF_MAG_APER_8_R_CORRECTED', 'PSF_MAG_ERR_APER_8_R',
    'EXT_XGB',
]

# ── edit these to choose your region ─────────────────────────────────────────
MY_RA_MIN, MY_RA_MAX   = 5, 42
MY_DEC_MIN, MY_DEC_MAX = -42, -14
# ─────────────────────────────────────────────────────────────────────────────

my_cat = lsdb.open_catalog(
    CATALOG_URL,
    search_filter=lsdb.BoxSearch(ra=(MY_RA_MIN, MY_RA_MAX), dec=(MY_DEC_MIN, MY_DEC_MAX)),
    columns=COLUMNS,
).compute()

my_cat = my_cat[
    (my_cat['EXT_XGB'] == 0) &
    (my_cat['PSF_MAG_APER_8_G_CORRECTED'] > 16) &
    (my_cat['PSF_MAG_APER_8_G_CORRECTED'] < 24)
].copy()
my_cat['g']     = my_cat['PSF_MAG_APER_8_G_CORRECTED']
my_cat['r']     = my_cat['PSF_MAG_APER_8_R_CORRECTED']
my_cat['color'] = my_cat['g'] - my_cat['r']
print(f'Downloaded {len(my_cat):,} stars')

In [ ]:
# ── isochrone / filter params — defaults reuse your tutorial selections ──────
MY_AGE  = BEST_AGE
MY_Z    = BEST_Z
MY_C    = BEST_C
MY_E    = BEST_E
MY_GMIN = BEST_GMIN
MY_GMAX = BEST_GMAX

MY_NSIDE = 256
MY_FWHM  = 0.4   # deg

# ── projection (derived from the coordinate box above) ───────────────────────
MY_RA_CTR  = (MY_RA_MIN + MY_RA_MAX) / 2
MY_DEC_CTR = (MY_DEC_MIN + MY_DEC_MAX) / 2
MY_DLON    = (MY_RA_MAX  - MY_RA_MIN)  / 2
MY_DLAT    = (MY_DEC_MAX - MY_DEC_MIN) / 2

# ── distance-modulus ranges for the three color channels (red=far) ────────────
MY_MU_R = (17.25, 18.5)   # far
MY_MU_G = (16.5,  17.0)   # mid
MY_MU_B = (15.5,  16.25)  # close
# ─────────────────────────────────────────────────────────────────────────────

_my_proj = hp.projector.CartesianProj(
    rot=[MY_RA_CTR, MY_DEC_CTR, 0], xsize=900, ysize=450,
    lonra=[-MY_DLON, MY_DLON], latra=[-MY_DLAT, MY_DLAT],
)

def _my_channel(mu_lo, mu_hi):
    stacked = np.zeros(hp.nside2npix(MY_NSIDE))
    for mu in np.arange(mu_lo, mu_hi + 0.01, 0.25):
        if mu > mu_hi + 1e-9:
            break
        mask = su.apply_matched_filter(my_cat, ISO_DIR, MY_AGE, MY_Z, mu,
                                       C=MY_C, E=MY_E, gmin=MY_GMIN, gmax=MY_GMAX)
        ipix = hp.ang2pix(MY_NSIDE,
                          my_cat['RA'].values[mask], my_cat['DEC'].values[mask],
                          lonlat=True, nest=True)
        np.add.at(stacked, ipix, 1)
    ring     = hp.reorder(stacked.astype(float), n2r=True)
    smoothed = hp.smoothing(ring, fwhm=np.radians(MY_FWHM))
    img = _my_proj.projmap(smoothed,
                           vec2pix_func=lambda x, y, z: hp.vec2pix(MY_NSIDE, x, y, z))
    return su.normalize_channel(img, lo=1, hi=99.5)

print('Building RGB channels...', end='', flush=True)
my_r = _my_channel(*MY_MU_R); print(' R', end='', flush=True)
my_g = _my_channel(*MY_MU_G); print(' G', end='', flush=True)
my_b = _my_channel(*MY_MU_B); print(' B  done.')

In [ ]:
my_rgb = np.dstack([my_r, my_g, my_b])

fig, ax = plt.subplots(figsize=(14, 7), facecolor='k')
ax.imshow(my_rgb, origin='lower', aspect='auto', interpolation='bilinear',
          extent=[MY_RA_CTR + MY_DLON, MY_RA_CTR - MY_DLON,
                  MY_DEC_CTR - MY_DLAT, MY_DEC_CTR + MY_DLAT])
ax.set_facecolor('k')
ax.set_xlabel('RA (deg)', fontsize=12, color='w')
ax.set_ylabel('Dec (deg)', fontsize=12, color='w')
ax.tick_params(labelsize=11, colors='w')
for spine in ax.spines.values():
    spine.set_edgecolor('w')
ax.set_xlim(MY_RA_CTR + MY_DLON, MY_RA_CTR - MY_DLON)
ax.set_ylim(MY_DEC_CTR - MY_DLAT, MY_DEC_CTR + MY_DLAT)
ax.set_title(
    'RGB distance map — custom region\n'
    f'R: μ={MY_MU_R[0]}–{MY_MU_R[1]}   '
    f'G: μ={MY_MU_G[0]}–{MY_MU_G[1]}   '
    f'B: μ={MY_MU_B[0]}–{MY_MU_B[1]}',
    color='w', fontsize=11,
)
plt.tight_layout()
plt.show()